In [0]:
# Databricks notebook: 04_gold_tables
from pyspark.sql import functions as F

silver = spark.table("estudo_estrangeiros.silver.player_season_club")

season_totals = (
    silver.groupBy("season")
    .agg(F.sum("minutes").alias("season_total_minutes"))
)

# --- Gold: age_nationality_minutes (a matriz da pergunta 3 do objetivo) ---
age_nationality_minutes = (
    silver.groupBy("season", "is_brazilian", "age_band")
    .agg(
        F.sum("minutes").alias("total_minutes"),
        F.countDistinct("player_id").alias("players"),
    )
    .join(season_totals, "season")
    .withColumn("pct_of_season_minutes", F.round(F.col("total_minutes") / F.col("season_total_minutes") * 100, 2))
    .drop("season_total_minutes")
    .orderBy("season", "is_brazilian", "age_band")
)

age_nationality_minutes.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.gold.age_nationality_minutes"
)
display(age_nationality_minutes)

# --- Gold: foreign_players (perguntas 1 e 5) ---
season_goals_assists = (
    silver.groupBy("season")
    .agg(
        F.sum("goals").alias("season_total_goals"),
        F.sum("assists").alias("season_total_assists"),
    )
)

foreign_players = (
    silver.groupBy("season", "nation_code", "is_brazilian")
    .agg(
        F.countDistinct("player_id").alias("players"),
        F.sum("minutes").alias("total_minutes"),
        F.sum("goals").alias("total_goals"),
        F.sum("assists").alias("total_assists"),
    )
    .join(season_totals, "season")
    .join(season_goals_assists, "season")
    .withColumn("pct_of_season_minutes", F.round(F.col("total_minutes") / F.col("season_total_minutes") * 100, 2))
    .withColumn("pct_of_season_goals", F.round(F.col("total_goals") / F.col("season_total_goals") * 100, 2))
    .withColumn("pct_of_season_assists", F.round(F.col("total_assists") / F.col("season_total_assists") * 100, 2))
    .withColumn(
        "goals_assists_per90",
        F.round((F.col("total_goals") + F.col("total_assists")) / (F.col("total_minutes") / 90), 3)
    )
    .drop("season_total_minutes", "season_total_goals", "season_total_assists")
    .orderBy("season", F.col("total_minutes").desc())
)

foreign_players.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.gold.foreign_players"
)
display(foreign_players)


season,is_brazilian,age_band,total_minutes,players,pct_of_season_minutes
2014,false,24-29,34548,33,4.6
2014,false,30+,24258,17,3.23
2014,false,<=23,7570,12,1.01
2014,true,24-29,342903,264,45.67
2014,true,30+,160823,126,21.42
2014,true,<=23,180735,276,24.07
2015,false,24-29,22780,28,3.04
2015,false,30+,18951,18,2.53
2015,false,<=23,4782,9,0.64
2015,true,24-29,328239,269,43.75


season,nation_code,is_brazilian,players,total_minutes,total_goals,total_assists,pct_of_season_minutes,pct_of_season_goals,pct_of_season_assists,goals_assists_per90
2014,BRA,true,666,684461,703,473,91.16,84.5,85.53,0.155
2014,ARG,false,22,26479,56,41,3.53,6.73,7.41,0.33
2014,PAR,false,11,11047,8,6,1.47,0.96,1.08,0.114
2014,CHI,false,4,6204,6,12,0.83,0.72,2.17,0.261
2014,URU,false,8,4100,1,3,0.55,0.12,0.54,0.088
2014,PER,false,2,3632,13,6,0.48,1.56,1.08,0.471
2014,BUL,false,1,2806,0,1,0.37,0.0,0.18,0.032
2014,BOL,false,1,2498,15,3,0.33,1.8,0.54,0.649
2014,QAT,false,2,1742,7,1,0.23,0.84,0.18,0.413
2014,ITA,false,2,1686,1,2,0.22,0.12,0.36,0.16


In [0]:
### Gold: club_performance

club_agg = (
    silver.groupBy("season", "squad")
    .agg(
        F.sum("minutes").alias("club_total_minutes"),
        F.sum(F.when(~F.col("is_brazilian"), F.col("minutes")).otherwise(0)).alias("foreign_minutes"),
        F.sum(F.when((F.col("is_brazilian")) & (F.col("age_band") == "<=23"), F.col("minutes")).otherwise(0)).alias("brazilian_u23_minutes"),
    )
    .withColumn("pct_minutes_foreign", F.round(F.col("foreign_minutes") / F.col("club_total_minutes") * 100, 2))
    .withColumn("pct_minutes_brazilian_u23", F.round(F.col("brazilian_u23_minutes") / F.col("club_total_minutes") * 100, 2))
    .select("season", "squad", "pct_minutes_foreign", "pct_minutes_brazilian_u23")
)

league = spark.table("estudo_estrangeiros.silver.league_table")

club_performance = league.join(club_agg, on=["season", "squad"], how="left").orderBy("position")

# checagem importante: nomes de clube têm que bater entre as duas tabelas
sem_match = club_performance.filter(F.col("pct_minutes_foreign").isNull())
print(f"Clubes sem correspondência entre league_table e player_season_club: {sem_match.count()}")
display(sem_match)

club_performance.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.gold.club_performance"
)
display(club_performance)


Clubes sem correspondência entre league_table e player_season_club: 0


season,squad,position,matches_played,wins,draws,losses,goals_for,goals_against,goal_diff,points,points_per_match,pct_minutes_foreign,pct_minutes_brazilian_u23


season,squad,position,matches_played,wins,draws,losses,goals_for,goals_against,goal_diff,points,points_per_match,pct_minutes_foreign,pct_minutes_brazilian_u23
2022,Palmeiras,1,38,23,12,3,66,27,39,81,2.13,17.74,18.14
2026,Flamengo,1,28,18,6,4,55,23,32,60,2.14,40.24,4.6
2020,Flamengo,1,38,21,8,9,68,48,20,71,1.87,12.41,29.54
2017,Corinthians,1,38,21,9,8,50,30,20,72,1.89,15.21,27.7
2018,Palmeiras,1,38,23,11,4,64,26,38,80,2.11,7.02,1.76
2025,Flamengo,1,38,23,10,5,78,27,51,79,2.08,37.95,9.06
2019,Flamengo,1,38,28,6,4,86,37,49,90,2.37,17.83,21.74
2015,Corinthians,1,38,24,9,5,71,31,40,81,2.13,3.76,11.85
2023,Palmeiras,1,38,20,10,8,64,33,31,70,1.84,20.62,19.56
2016,Palmeiras,1,38,24,8,6,62,32,30,80,2.11,4.57,20.27


In [0]:
from pyspark.sql import Row

regras_limite_estrangeiros = spark.createDataFrame([
    Row(season=2014, limite_estrangeiros_por_partida=5),
    Row(season=2015, limite_estrangeiros_por_partida=5),
    Row(season=2016, limite_estrangeiros_por_partida=5),
    Row(season=2017, limite_estrangeiros_por_partida=5),
    Row(season=2018, limite_estrangeiros_por_partida=5),
    Row(season=2019, limite_estrangeiros_por_partida=5),
    Row(season=2020, limite_estrangeiros_por_partida=5),
    Row(season=2021, limite_estrangeiros_por_partida=5),
    Row(season=2022, limite_estrangeiros_por_partida=5),
    Row(season=2023, limite_estrangeiros_por_partida=7),
    Row(season=2024, limite_estrangeiros_por_partida=9),
    Row(season=2025, limite_estrangeiros_por_partida=9),
    Row(season=2026, limite_estrangeiros_por_partida=9),
])

regras_limite_estrangeiros.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    "estudo_estrangeiros.gold.regras_limite_estrangeiros"
)
display(regras_limite_estrangeiros)


season,limite_estrangeiros_por_partida
2014,5
2015,5
2016,5
2017,5
2018,5
2019,5
2020,5
2021,5
2022,5
2023,7
